# 08 Direct Model Calibration, Refit, and Test

**Objective:** train one explicitly chosen Ultralytics detector without a
candidate sweep: first calibrate its stopping epoch on frozen `train`/`val`,
then restart from pretrained weights and refit on `train + val` for exactly the
locked epoch count before one independent test evaluation.

**Success criteria:** model and source policy are user-selected before test,
the refit epoch is validation-derived, test data never influences that choice,
and only the final refit checkpoint plus compact deployment artifacts remain.


## Setup, model choice, and test authorization

Set `MODEL_NAME` to any Ultralytics-supported checkpoint or local model path.
`yolo12n.pt` and `yolo12s.pt` are convenient presets, not a candidate list.

`USE_OPEN_SOURCE_DATA=False` keeps both phases factory-only. Online
augmentation uses the same tracked arguments as Notebook 04; offline-generated
data remains outside this direct original-data workflow.

Set `TRAIN_IMGSZ` and `TRAIN_BATCH` explicitly when this optional direct path needs a non-baseline resolution.

A real calibration → refit → test run requires `DRY_RUN=False` and
`ALLOW_TEST_EVALUATION=True`. Decide all options before viewing test metrics.


In [ ]:
# Purpose: initialize one direct two-phase run without candidate or ablation dependencies.
# Inputs: shared Notebook 01/02 data contract, Notebook 04 runtime settings, and explicit user options.
# Guard: default to dry-run; require explicit test authorization and a completed frozen split artifact.
# Output: one immutable direct-final artifact with source policy, runtime, and upstream provenance.
from pathlib import Path
import sys

BOOTSTRAP_ROOT = Path.cwd().resolve()
if BOOTSTRAP_ROOT.name == "notebooks":
    BOOTSTRAP_ROOT = BOOTSTRAP_ROOT.parent
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.artifacts import (
    create_run,
    find_latest_run,
    open_run,
    read_run_metadata,
    runtime_metadata,
    update_run_metadata,
    write_run_metadata,
)
from src.config import (
    find_project_root,
    load_pipeline_config,
    online_augmentation_args,
)
from src.dataset.manifests import (
    build_development_training_manifest,
    build_original_training_manifest,
    copy_manifest,
    manifest_pair_content_sha256,
    manifest_sha256,
    read_manifest,
    write_ultralytics_dataset_yaml,
)
from src.training.evaluate_yolo import (
    evaluate_model,
    extract_best_epoch,
    save_confusion_matrix,
    save_prediction_grid,
)
from src.training.export_model import export_final_model
from src.training.train_yolo import (
    build_train_args,
    clean_training_intermediates,
    format_progress_event,
    retain_training_checkpoint,
    run_training_specs,
)

V2_ROOT = find_project_root(BOOTSTRAP_ROOT)
CONFIG = load_pipeline_config(V2_ROOT)
TRAINING_CONFIG = CONFIG["training"]
SEED = int(TRAINING_CONFIG["seed"])

# User options: MODEL_NAME is free-form; presets only document known convenient choices.
MODEL_PRESETS = ("yolo12n.pt", "yolo12s.pt")
MODEL_NAME = "yolo12n.pt"
USE_OPEN_SOURCE_DATA = False
USE_ONLINE_AUGMENTATION = True
# Explicit direct-path resolution; defaults preserve the tracked baseline.
TRAIN_IMGSZ = int(TRAINING_CONFIG["imgsz"])
TRAIN_BATCH = int(TRAINING_CONFIG["batch"])
SOURCE_SPLIT_RUN = (
    None  # Prefer an explicit Notebook 02 artifact for a final recorded run.
)
DRY_RUN = True
ALLOW_TEST_EVALUATION = False

if not str(MODEL_NAME).strip():
    raise ValueError("MODEL_NAME must be a checkpoint name or local model path")
if not DRY_RUN and not ALLOW_TEST_EVALUATION:
    raise RuntimeError(
        "Direct calibration/refit/test requires ALLOW_TEST_EVALUATION=True"
    )

SPLIT_RUN = (
    open_run(SOURCE_SPLIT_RUN)
    if SOURCE_SPLIT_RUN
    else find_latest_run(V2_ROOT, "split-manifests", "manifests/split_assignments.csv")
)
split_metadata = read_run_metadata(SPLIT_RUN)
if split_metadata.get("status") != "completed":
    raise RuntimeError(f"Split run is not completed: {SPLIT_RUN.run_id}")
expected_test_content_sha256 = split_metadata.get("test_pair_content_sha256")
if not expected_test_content_sha256:
    raise RuntimeError("Split metadata lacks the frozen test content fingerprint")

data_scope = "factory-open" if USE_OPEN_SOURCE_DATA else "factory-only"
RUN = create_run(
    V2_ROOT,
    "direct-final",
    f"{Path(str(MODEL_NAME)).stem}-{data_scope}-i{TRAIN_IMGSZ}-calibrated",
    SEED,
)
write_run_metadata(
    RUN,
    {
        "stage": "direct-final",
        "status": "started",
        "dry_run": DRY_RUN,
        "test_evaluation_authorized": ALLOW_TEST_EVALUATION,
        "seed": SEED,
        "model_name": str(MODEL_NAME),
        "model_presets": list(MODEL_PRESETS),
        "use_open_source_data": USE_OPEN_SOURCE_DATA,
        "use_online_augmentation": USE_ONLINE_AUGMENTATION,
        "train_imgsz": TRAIN_IMGSZ,
        "train_batch": TRAIN_BATCH,
        "epoch_policy": "calibrate_on_train_val_then_refit_train_plus_val",
        "source_split_run": SPLIT_RUN.run_id,
        "source_audit_run": split_metadata.get("source_audit_run"),
        "runtime": runtime_metadata(V2_ROOT),
    },
)


## Materialize calibration, refit, and test membership

Phase 1 trains on frozen `train` and validates on the disjoint factory `val`
manifest. Phase 2 combines those original memberships into `train + val`.
When open-source data is disabled, only its train rows are removed; every
factory row remains in the appropriate phase.

The calibration YAML intentionally omits test. The refit YAML includes the
test manifest for later evaluation, but training validation and early stopping
will be disabled during refit. Test image-label bytes must match Notebook 02's
frozen fingerprint before either phase starts.


In [ ]:
# Purpose: create exact phase-specific manifests and verify untouched test integrity.
# Inputs: frozen split assignments, source toggle, four-class map, and test fingerprint.
# Guard: keep calibration val disjoint, never add offline data, and reject missing files or test overlap.
# Output: two dataset YAMLs plus compact phase/source membership and locked-setting summaries.
assignments_path = SPLIT_RUN.manifests / "split_assignments.csv"
calibration_train_manifest = build_original_training_manifest(
    assignments_path,
    RUN.manifests / "calibration_train_original.txt",
    include_open_source=USE_OPEN_SOURCE_DATA,
    include_validation=False,
)
calibration_val_manifest = copy_manifest(
    SPLIT_RUN.manifests / "val.txt",
    RUN.manifests / "calibration_val.txt",
)
development_manifest = build_development_training_manifest(
    assignments_path,
    RUN.manifests / "refit_train_plus_val_original.txt",
    include_open_source=USE_OPEN_SOURCE_DATA,
)
test_manifest = copy_manifest(
    SPLIT_RUN.manifests / "test.txt",
    RUN.manifests / "test.txt",
)

# Hash both test images and labels before training; changed test bytes abort the new run.
test_content_sha256 = manifest_pair_content_sha256(test_manifest)
if test_content_sha256 != expected_test_content_sha256:
    raise RuntimeError(
        "Untouched test image/label content changed after split freezing"
    )

calibration_dataset_yaml = write_ultralytics_dataset_yaml(
    RUN.root / "calibration_dataset.yaml",
    calibration_train_manifest,
    calibration_val_manifest,
    CONFIG["classes"],
)
refit_dataset_yaml = write_ultralytics_dataset_yaml(
    RUN.root / "refit_dataset.yaml",
    development_manifest,
    development_manifest,
    CONFIG["classes"],
    test_manifest=test_manifest,
    allow_train_val_overlap=True,
)

membership_df = pd.DataFrame(
    [
        {
            "phase": "calibration train",
            "manifest": calibration_train_manifest.name,
            "images": len(read_manifest(calibration_train_manifest)),
        },
        {
            "phase": "calibration val",
            "manifest": calibration_val_manifest.name,
            "images": len(read_manifest(calibration_val_manifest)),
        },
        {
            "phase": "locked-epoch refit",
            "manifest": development_manifest.name,
            "images": len(read_manifest(development_manifest)),
        },
        {
            "phase": "untouched test",
            "manifest": test_manifest.name,
            "images": len(read_manifest(test_manifest)),
        },
    ]
)
locked_settings_df = pd.DataFrame(
    [
        {
            "model": str(MODEL_NAME),
            "available_presets": ", ".join(MODEL_PRESETS),
            "open_source": USE_OPEN_SOURCE_DATA,
            "online_augmentation": USE_ONLINE_AUGMENTATION,
            "offline_augmentation": False,
            "imgsz": TRAIN_IMGSZ,
            "batch": TRAIN_BATCH,
            "calibration_max_epochs": int(TRAINING_CONFIG["final_epochs"]),
            "calibration_patience": int(TRAINING_CONFIG["patience"]),
            "dry_run": DRY_RUN,
        }
    ]
)
display(locked_settings_df)
display(membership_df)


#### Observation

TODO after execution:

- Confirm the selected model, source policy, and augmentation policy.
- Confirm calibration train/val are disjoint and refit contains their intended union.
- Confirm offline augmentation is absent and the test fingerprint is unchanged.


## Phase 1 — calibrate the stopping epoch on validation

Train only the selected model from pretrained weights on calibration train.
Validation runs every epoch with configured `patience`; no test path exists in
this phase's dataset YAML. Lock the first epoch with maximum validation
mAP50-95, matching Ultralytics detection fitness and avoiding test-driven epoch
selection.


In [ ]:
# Purpose: estimate the stopping epoch for this exact model, source policy, and online augmentation policy.
# Inputs: disjoint calibration train/val manifests and shared final-training runtime arguments.
# Guard: calibrate one model only, permit configured early stopping, and keep test absent from dataset YAML.
# Output: one live validation panel and one compact summary containing the locked best epoch.
calibration_train_args = build_train_args(TRAINING_CONFIG, "final_epochs")
# Keep calibration and refit at the same explicit direct-path resolution.
calibration_train_args.update({"imgsz": TRAIN_IMGSZ, "batch": TRAIN_BATCH})
calibration_spec = {
    "run_name": f"calibration-{Path(str(MODEL_NAME)).stem}-{data_scope}",
    "model_name": str(MODEL_NAME),
    "experiment": "direct_epoch_calibration",
    "data_yaml": calibration_dataset_yaml,
    "online_augmentation": USE_ONLINE_AUGMENTATION,
    "augmentation_args": online_augmentation_args(
        CONFIG["augmentation"],
        enabled=USE_ONLINE_AUGMENTATION,
    ),
}
calibration_progress = display(
    Markdown("Epoch calibration is waiting to start."),
    display_id=True,
)


def update_calibration_progress(event: dict) -> None:
    """Render one compact calibration event in the existing panel."""
    calibration_progress.update(Markdown(format_progress_event(event)))


calibration_summary_df = run_training_specs(
    [calibration_spec],
    RUN.models,
    calibration_train_args,
    dry_run=DRY_RUN,
    continue_on_error=False,
    progress_callback=update_calibration_progress,
)
if DRY_RUN:
    epoch_selection = {
        "best_epoch": int(TRAINING_CONFIG["final_epochs"]),
        "selection_metric": "dry_run_fallback",
        "precision": None,
        "recall": None,
        "map50": None,
        "map50_95": None,
    }
    epoch_selection_source = "dry_run_fallback"
else:
    calibration_run_dir = Path(str(calibration_summary_df.iloc[0]["run_dir"]))
    epoch_selection = extract_best_epoch(calibration_run_dir)
    epoch_selection_source = "calibration_validation_map50_95"

locked_refit_epochs = int(epoch_selection["best_epoch"])
for key, value in epoch_selection.items():
    calibration_summary_df.loc[0, f"selected_{key}"] = value
calibration_summary_path = RUN.csv / "calibration_epoch_selection.csv"
calibration_summary_df.to_csv(calibration_summary_path, index=False)
display(
    calibration_summary_df[
        [
            "model_name",
            "status",
            "selected_best_epoch",
            "selected_recall",
            "selected_map50",
            "selected_map50_95",
            "training_time_seconds",
        ]
    ]
)


#### Observation

TODO after execution:

- Record the locked best epoch and its validation mAP50-95.
- Note whether early stopping shortened the 100-epoch calibration budget.
- Do not alter the locked epoch after any later test result.


## Phase 2 — refit train + val for the locked epoch count

Restart the same architecture from its original pretrained checkpoint; do not
continue from calibration `best.pt`. Fit every selected development image for
exactly `locked_refit_epochs`. Validation and early stopping are disabled, and
the final retained checkpoint is `last.pt` because the stopping decision was
already made by Phase 1.


In [ ]:
# Purpose: refit the selected architecture on all development images for the validation-locked epoch count.
# Inputs: train-plus-val manifest, locked epoch, shared runtime settings, and the same online augmentation policy.
# Guard: restart from pretrained MODEL_NAME, disable validation/early stopping, and never adjust epochs from test.
# Output: one loss-only progress panel and one compact fixed-epoch refit summary.
refit_train_args = build_train_args(TRAINING_CONFIG, "final_epochs")
refit_train_args.update(
    {
        "epochs": locked_refit_epochs,
        "imgsz": TRAIN_IMGSZ,
        "batch": TRAIN_BATCH,
        "val": False,
        "patience": 0,
    }
)
refit_spec = {
    "run_name": f"refit-{Path(str(MODEL_NAME)).stem}-{data_scope}-e{locked_refit_epochs}",
    "model_name": str(MODEL_NAME),
    "experiment": "direct_locked_epoch_refit",
    "data_yaml": refit_dataset_yaml,
    "online_augmentation": USE_ONLINE_AUGMENTATION,
    "augmentation_args": online_augmentation_args(
        CONFIG["augmentation"],
        enabled=USE_ONLINE_AUGMENTATION,
    ),
}
refit_progress = display(
    Markdown("Locked-epoch refit is waiting to start."),
    display_id=True,
)


def update_refit_progress(event: dict) -> None:
    """Render one compact loss-only refit event in the existing panel."""
    refit_progress.update(Markdown(format_progress_event(event)))


refit_summary_df = run_training_specs(
    [refit_spec],
    RUN.models,
    refit_train_args,
    dry_run=DRY_RUN,
    continue_on_error=False,
    progress_callback=update_refit_progress,
)
refit_summary_df.insert(2, "locked_refit_epochs", locked_refit_epochs)
refit_summary_path = RUN.csv / "locked_epoch_refit_summary.csv"
refit_summary_df.to_csv(refit_summary_path, index=False)
display(
    refit_summary_df[
        [
            "model_name",
            "status",
            "locked_refit_epochs",
            "training_time_seconds",
            "model_size_mb",
            "error_message",
        ]
    ]
)


#### Observation

TODO after execution:

- Confirm refit completed exactly the validation-locked epoch count.
- Treat any final seen-data metric as diagnostic only; it did not select the checkpoint.


## Retain the refit checkpoint and evaluate test once

Copy refit `last.pt` to a descriptive stable model file, then evaluate
`split=test`. Save compact overall/per-class metrics, one confusion matrix, and
one bounded six-image prediction grid. Test results are reporting-only.


In [ ]:
# Purpose: retain the locked-epoch refit and perform the explicitly authorized test evaluation.
# Inputs: successful refit last.pt, untouched test manifest, image size, device, and reporting confidence.
# Guard: dry-run skips test metrics; test results cannot change model, source policy, augmentation, or epoch count.
# Output: retained model, compact test CSVs, confusion matrix, and six-image prediction sample.
final_checkpoint = None
test_overall_df = pd.DataFrame()
test_class_metrics_df = pd.DataFrame()
confusion_figure_path = None
prediction_figure_path = None
if DRY_RUN:
    print("Dry run: checkpoint retention and test evaluation skipped.")
else:
    final_checkpoint = retain_training_checkpoint(
        refit_summary_df,
        RUN.models
        / f"{Path(str(MODEL_NAME)).stem}_calibrated-e{locked_refit_epochs}_train-plus-val.pt",
        checkpoint_kind="last",
    )
    evaluation = evaluate_model(
        final_checkpoint,
        refit_dataset_yaml,
        split="test",
        imgsz=TRAIN_IMGSZ,
        device=TRAINING_CONFIG["device"],
    )
    test_overall_df = pd.DataFrame([evaluation["overall"]])
    test_class_metrics_df = evaluation["class_metrics"]
    test_overall_df.to_csv(RUN.csv / "direct_test_overall_metrics.csv", index=False)
    if not test_class_metrics_df.empty:
        test_class_metrics_df.to_csv(
            RUN.csv / "direct_test_class_metrics.csv", index=False
        )
    if evaluation["confusion_matrix"] is not None:
        confusion_figure_path = save_confusion_matrix(
            evaluation["confusion_matrix"],
            CONFIG["classes"].values(),
            RUN.figures / "direct_test_confusion_matrix.png",
        )
    prediction_figure_path = save_prediction_grid(
        final_checkpoint,
        test_manifest,
        RUN.figures / "direct_test_prediction_samples.png",
        imgsz=TRAIN_IMGSZ,
        device=TRAINING_CONFIG["device"],
        conf=float(TRAINING_CONFIG["final_prediction_conf"]),
        max_images=6,
    )
    display(test_overall_df)
    display(test_class_metrics_df)


#### Observation

TODO after execution:

- Report overall test precision, recall, mAP50, and mAP50-95.
- Identify weak PPE classes and visible failure modes from the bounded outputs.
- Do not rerun with a different epoch/model/source toggle based on this test result.


## Export and apply retention policy

Export only configured deployment formats. Keep the descriptive locked-epoch
refit checkpoint, then remove this new run's calibration/refit `_work` and
shared `_pretrained` directories when intermediate retention is disabled.


In [ ]:
# Purpose: export the retained refit model and remove redundant current-run internals.
# Inputs: retained checkpoint, configured export formats, and keep_intermediates policy.
# Guard: export failures are reported without deleting the retained PyTorch checkpoint.
# Output: optional deployment model(s), one compact export CSV, and a clean models folder.
export_report_df = pd.DataFrame()
export_summary_path = None
if not DRY_RUN and final_checkpoint is not None:
    export_report_df = export_final_model(
        final_checkpoint,
        TRAINING_CONFIG["final_export_formats"],
        RUN.models,
    )
    export_summary_path = RUN.csv / "direct_export_summary.csv"
    export_report_df.to_csv(export_summary_path, index=False)
    if not bool(TRAINING_CONFIG["keep_intermediates"]):
        clean_training_intermediates(RUN.models)
    display(export_report_df)


#### Observation

TODO after execution: record retained checkpoint size, deployment format
availability, and any machine-dependent export failure.


## Finalize the immutable calibrated-refit report

Persist the validation-derived epoch decision, source policy, both manifest
contracts, test metrics, and retained paths in this run's `run.yaml`.


In [ ]:
# Purpose: close the direct run with auditable calibration, refit, test, and artifact evidence.
# Inputs: frozen manifests, selected epoch, compact summaries, test results, figures, and exports.
# Guard: update only this newly created run and never reinterpret or overwrite a previous test run.
# Output: final run.yaml plus concise run ID, locked epoch, and completion status.
update_run_metadata(
    RUN,
    {
        "status": "dry_run" if DRY_RUN else "completed",
        "model_name": str(MODEL_NAME),
        "epoch_policy": "calibrate_on_train_val_then_refit_train_plus_val",
        "epoch_selection_source": epoch_selection_source,
        "locked_refit_epochs": locked_refit_epochs,
        "epoch_selection_validation_metrics": {
            key: value
            for key, value in epoch_selection.items()
            if key not in {"best_epoch", "selection_metric"}
        },
        "use_open_source_data": USE_OPEN_SOURCE_DATA,
        "use_online_augmentation": USE_ONLINE_AUGMENTATION,
        "train_imgsz": TRAIN_IMGSZ,
        "train_batch": TRAIN_BATCH,
        "use_offline_augmentation": False,
        "calibration_dataset_yaml": str(calibration_dataset_yaml),
        "refit_dataset_yaml": str(refit_dataset_yaml),
        "calibration_train_manifest_sha256": manifest_sha256(
            calibration_train_manifest
        ),
        "calibration_val_manifest_sha256": manifest_sha256(calibration_val_manifest),
        "refit_development_manifest_sha256": manifest_sha256(development_manifest),
        "test_manifest_sha256": manifest_sha256(test_manifest),
        "test_pair_content_sha256": test_content_sha256,
        "calibration_summary_csv": str(calibration_summary_path),
        "refit_summary_csv": str(refit_summary_path),
        "final_checkpoint": str(final_checkpoint) if final_checkpoint else None,
        "test_overall_metrics": test_overall_df.to_dict(orient="records"),
        "test_class_metrics_csv": str(RUN.csv / "direct_test_class_metrics.csv")
        if not test_class_metrics_df.empty
        else None,
        "confusion_matrix_figure": str(confusion_figure_path)
        if confusion_figure_path
        else None,
        "prediction_samples_figure": str(prediction_figure_path)
        if prediction_figure_path
        else None,
        "export_summary_csv": str(export_summary_path) if export_summary_path else None,
    },
)
print(f"Direct calibrated run: {RUN.run_id}")
print(f"Locked refit epochs: {locked_refit_epochs}")
print(f"Status: {'dry_run' if DRY_RUN else 'completed'}")


#### Observation

TODO after execution: write the exact immutable run ID, locked refit epoch,
deployment recommendation, known limitations, and whether this source policy
should remain fixed for future production retraining.
